# Trace two bids without a detailed BOQ — `N`/unconfirmed vs. `null`

Unity Catalog: `ingestion_framework_test.bid_data_exploration`

Contrasts **D-111808** — our established real construction tender, already known from `FINDINGS.md`'s headline finding to have only one lump-sum `quotationline` row per vendor, no itemized breakdown — against a genuinely **`null`-flagged** example, tracing both through every table the same way `07_trace_bid_with_boq.ipynb` does for a `Y` example.

This also finally answers the still-open question from `01_rfq.ipynb`'s Run 3: what does D-111808's own `DETAILBOQAVAILABLE` value actually say? If it turns out to be `N`, that's a clean match to what we already know from its `quotationline` data. If both D-111808 and the `null` example look the same end-to-end, that's evidence `null` really does mean "no BOQ" in practice. If the `null` example actually has rich `quotationline` data, that's evidence `null` just means "never assessed", not "no BOQ" — directly resolving the open question flagged in `FINDINGS.md`.

## Part 1 — D-111808

Already partially traced in `02_rfqvendor.ipynb` (22 invited vendors), `03_quotationline.ipynb` (8 priced lump-sum lines), and `04_altquotationline.ipynb` (0 alternates). This section pulls everything into one place, plus the two pieces never checked yet: its actual `rfq` row (including the real `DETAILBOQAVAILABLE` value) and any attached documents.

### `rfq` — header (first time checking this specific row)

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq WHERE RFQNUM = 'D-111808'

RFQNUM,DESCRIPTION,STATUS,STATUSDATE,ENTERDATE,ENTERBY,REPLYDATE,CLOSEONDATE,PURCHASEAGENT,RFQTYPE,REQUIREDDATE,REQUESTEDBY,SHIPTO,SHIPTOATTN,BILLTO,BILLTOATTN,REPLYTO,REPLYTOATTN,FOB,FREIGHTTERMS,SHIPVIA,PAYMENTTERMS,CHANGEBY,CHANGEDATE,PRIORITY,HISTORYFLAG,RFQ1,RFQ2,RFQ3,RFQ4,RFQ5,RFQ6,RFQ7,RFQ8,RFQ9,RFQ10,PRINTDATE,BUYERCOMPANY,ORGID,SITEID,RFQID,LANGCODE,HASLD,TDCOMPLETED,INSURANCEREV,TOTALAWVALUE,PMETHOD,VLSUBMSN,LEGALREV,AWCOMPLETED,TDAGENDA,FINANCEREV,TDMEETING,TDDATE,EXTCLOSEDATE,COMCONT,VLDCSN,WARRANTY,CONTYPE,AWDCSN,CONTPHONE,AWAGENDA,AWDATE,USEREV,FIXEDASSETS,TOTALCOST,TDDECISION,AWDECISION,ESTIMATEDCOST,AWSUBMSN,AWMEETING,DELVDEST,PROJCONS,CURRENCYCODE,INSPCLASS,AFENUM,ESTIMATEVAL,REPNAME,APPTYPE,TYPE,ROWSTAMP,REMARK,WFREPREMARKS,BIDSOPENDATE,KPIREMARKS,MAXINDIVIDUALVOL,TENDERDOCAPPRDATE,TENDERDOCSENDDATE,KPI1DATE,KPI2DATE,KPI3DATE,KPI4DATE,KPI5DATE,KPI6DATE,KPI17ATE,KPI8DATE,KPI9DATE,KPI10DATE,KPI7DATE,KPI11DATE,FIRSTDRAFT,SECONDRAFT,IITPACK,KPI,BIDBOND,PERFBOND,BENEFICIARY,CAPEX_CATEGORY,DURATIONUNIT,FUND_SOURCE,TYPE_BUSINESS,WARRANTYDAY,WARRANTYUNIT,CC,CLARIFICATIONDATE,DEALTBY,FORREVIEW,NOTE,PLEASECOMMENT,PLEASERECYCLE,PLEASEREPLY,PRINTDRAFTCOPY,PRINTESIGNATURE,REFNO,SELECTREPORT,SITEVISIT,TENDERDATE,URGENT,VENDOR,CBIDSOPENDATE,EOICLOSINGDATE,EOIEMAILSUBJECT,EOIEMAILTEXT,EOIREMARKS,EOIUSED,TENDERSTATUS,USEEBIDDING,EOIREFNO,EXTPRICECLOSEDATE,PRICECLOSEDATE,BIDEXTREFNO,INVITEREFNO,PIREFNO,EVALUATIONCUTOFF,TOTALSCORE,TRAINREQ,INSTALLREQ,SAMPLEREQ,CONCREQ,CALCREQ,TESTREPREQ,SPACKINGREQ,TECHDRAWREQ,TRAINING,SPACKDETAILS,TRAININGDAY,TRAININGUNIT,DOWNLOADTYPE,BIDBONDVALUE,EOIPUBLISHDATE,PUBLISHQUOTE,BIDBONDVALIDITY,CEILINGVALUE,RFQAUTHCODE,RFQCOMAUTHCODE,ADVANCEPAYMENT,ADVANCEPAYMENTPERCENT,DETAILBOQAVAILABLE,RETENTIONAPPLIED,RETENTIONPERCENT,COMEVALCUTOFF,COMTOTALSCORE,PARTIALDELIVERY,JUSTIFICATION,CPCSTARTDATE,CPCENDDATE,ECSTARTDATE,ECENDDATE,TECHSTART,TECHEND,COMMSTART,COMMEND,CPCSTART,CPCEND,BLREPDATE,BLSDDATE,AWARDREPDATE,AWARDSDDATE,TOTALAWVALUEWITHTAX,SUBWORKTYPE,CONSREQUIRED,PERFBANK,RETENMONEY,RPAC,RFAC,ADVANCEPAY,INVITEENTERED,INVITETEMP,EOICLOSED,BIDCLOSED,ECSTART,ECEND,COMPDISQUALIFIED,TETC_ENDDATE,AWTC_ENDDATE,TETC_STARTDATE,AWTC_STARTDATE,BUDGETBALANCE,BUDGETAVAILABLE,BUDGETCAT,ESTBUDGET,GUIDNUM,ITRTYPE,PROJECTREV,PROJECTID,ITRNUM,BUDGETCODE,APMASTERREV,APMASTERID,EXT_PROJECTID,EXT_ITRNUM,EXT_BUDGETCODE,EXT_PROJECTREV,POSTBID_DISCOUNT_CLOSEDATE,DISCOUNT_REVISION,EXT_POSTBID_DISCOUNT_CLOSEDATE,RECALLREPUBLISH,ALLOWAMENDMENT,N_PROJECT_NUMBER,N_TASK_NUMBER,UNDERREVIEW,ADPCBOARD,ADPCENDORSE,ADPCEO,ADPCHEC,ADPCREVIEWERS,ADPCTPC,ASTTEAM,ISPROJECT,ESTIMATECOST,ALLOWSINGLEIMPORT,SINGLESOURCE,MEMBERSFORIMP,BP_MASTERMILESTONE,ENDDATE,STARTDATE,PLANNEDSTARTDATE,PROCESSSTARTDATE,AWD_REMARKS,COMEVAL_REMARKS,TECHEVAL_REMARKS,FIRSTCBIDSOPENDATE,SHAREDWITHEWEC,NBFP,NBFPDONE,PROJLIFECYCLEID,EXTAADC1,DURATIONDAYS,CATEGORY,TDCOMMENTS,TDENDORSEDATE,TDENTERBY,TDRESULT,COMOPENDATE,COMOPENUSER,STAGE,TECHOPENDATE,TECHOPENUSER,AUTOROUTE,BUDGENHCOMMENTS,BUDGENHENDORSEDATE,BUDGENHENTERBY,BUDGENHRESULT,BLACOMMENTS,BLAENDORSEDATE,BLATDENTERBY,BLARESULT,BUDGENHFMDATE,BUDGENHFMENTERBY,BUDGENHMDDATE,BUDGENHMDENTERBY,BUDGENHSMDATE,BUDGENHSMENTERBY,BLASMDATE,BLASMENTERBY,SAMPLETESTREQ,BUDGENHCOMMENTSFIN,BUDGENHFINOFF,BUDGENHRESULTFIN,BUDGENHDEPTDEC,BUDGENHFMDEC,BUDGENHMDDEC,BUDGENHREJBY,BUDGENHREJDATE,BUDGENHREJREAASON,BUDGENHSMDEC,PROJECTDUR,PROJECTDURUNIT,TD,FU_STATUS,FU_SOURCE,FU_STATUSBY,FU_STATUSDATE,TECHEVALREQ,N_ISTTH
D-111808,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",CLOSE,2024-02-13T10:10:56Z,2023-04-19T17:46:24Z,DE046454,2023-08-03T10:00:00Z,2023-06-22T10:00:00Z,DE046454,null,2023-11-15T00:00:00Z,DE045233,null,DE045233,null,null,null,null,null,null,null,null,DE046454,2024-02-13T10:10:57Z,1.0000000000,1.0000000000,null,null,null,null,24 Months + 24 Months warranty,null,null,0E-10,null,null,2023-05-30T00:00:00Z,null,ADDCORG,ADDC,105585.0000000000,EN,0E-10,0E-10,nul

### `rfqvendor` — invited vendors (22 expected)

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor WHERE RFQNUM = 'D-111808' ORDER BY VENDOR

RFQNUM,VENDOR,CONTACT,PHONE,FAXPHONE,EMAIL,CUSTOMERNUM,FOB,FREIGHTTERMS,SHIPVIA,PAYMENTTERMS,CURRENCYCODE,EXCHANGERATE,EXCHANGEDATE,BUYAHEAD,INTERNAL,REPLIEDDATE,GLCREDITACCT,INCLUSIVE1,INCLUSIVE2,INCLUSIVE3,INCLUSIVE4,INCLUSIVE5,VENDORQUOTENUM,ORGID,SITEID,RFQVENDORID,LANGCODE,HASLD,ROWSTAMP,BUYERREMARK,LINENUM,RFQV1,PRODUCTGROUPID,CURRENCYCODE2,RFQV3,RFQV4,DISCOUNT,SQR,MNGREMARK,ENTERDATE,RFQV2,OTHERCHG,RFQV5,SSUPREMARK,ENTEREDBY,ISVENINC,BYRRMK,SNSRMK,MGTRMK,BIDSTATUS,BIDSTATUSDATE,PASSWORD,PRICEIMPACT,REGRETREASON,RESETPASSWORD,CI,TCI,TI,USERID,VALIDTO,SUPPORTREMARKS,SUPPOERTCONTACTED,CIINITIATED,VALIDITYPERIOD,SCORE,SURVEYCREATED,BASETOTALAWARDCOST,TOTALAWARDCOST,TOTALDISCOUNT,TOTAWDWODISCOUNT,TOTBIDCOSTWDIS,TOTBIDCOSTWODIS,TOTALAWARDCOSTWITHTAX,COMPEVALTYPE,APPLYDISCOUNTBEFOREBIDS,APPLYDISCOUNTAFTERBIDS,DISCOUNT_PERCENT,APPLYDISCOUNT,DISCOUNTAFTERBIDS,APPLYDISCOUNTAFTERPI,DCI,DCIREQUIRED,DISCOUNTSTATUS,DISCOUNT_APPLIED_AFTERBIDS,DISCOUNT_APPLIED_AFTERPI,DISCOUNT_APPLIED_BEFOREBIDS,POSTBID_DISCOUNT_COUNTER,POSTBID_DISCOUNT_SENT,DISCOUNT_REVISION,DISCOUNT_SUBMISSION_DATE,TOTALAWARDCOSTWDIS,TOTALAWARDCOSTWITHTAXWDIS,DISCOUNT_APPLY_DATE,QUOTERENEWED,QUOTEVALIDITY,RFQV_EXTRA1,RFQV_EXTRA2,ISAWARDED,AGREECOC,WAIVEOWNERVAL
D-111808,001565,Hussam Saghir,0097126262800,0097126269871,null,null,null,null,null,null,AED,1.0000000,2023-04-24T00:00:00Z,0E-10,0E-10,2023-08-03T09:58:56Z,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,L1352,ADDCORG,ADDC,527692.0000000000,EN,0E-10,12167668245,null,null,null,null,null,1.0000000000,null,null,null,null,2023-04-24T09:58:37Z,null,null,null,null,MAXREG,0E-10,null,null,9.0000000000,SUBMITTED,2023-08-03T09:58:56Z,null,0E-10,null,0E-10,1.0000000000,0E-10,1.0000000000,001565LGAU,null,null,null,0E-10,180,null,0E-10,CGTqea+foUw=,CGTqea+foUw=,null,CGTqea+foUw=,71kK+VMds2CuOLh7DkZdng==,GXYG+V/Gtzy+MLESDx0o0Q==,CGTqea+foUw=,null,1.0000000000,0E-10,0.00,null,0E-10,0E-10,1.0000000000,0E-10,PostBid Discount Submiited,1.0000000000,0E-10,0E-10,2.0000000000,0E-10,2.0000000000,2023-10-26T09:57:41Z,0.00,0.00,2023-10-20T14:41:12Z,0E-10,2023-12-31T09:53:39Z,null,null,0E-10,0E-10,0E-10
D-111808,001588,Abdul Aziz Chohan,0097126788621,0097126783085,null,null,null,null,null,null,AED,1.0000000,2023-04-24T00:00:00Z,0E-10,0E-10,null,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,AA/TEC/788,ADDCORG,ADDC,527703.0000000000,EN,0E-10,11490118273,null,null,null,null,null,1.0000000000,null,null,null,null,2023-04-24T11:48:55Z,null,null,null,null,MAXREG,0E-10,null,null,12.0000000000,COLLECTED,2023-05-30T19:09:11Z,null,0E-10,null,0E-10,0E-10,0E-10,0E-10,001588L85M,null,null,null,0E-10,120,null,0E-10,null,null,null,null,null,null,null,null,0E-10,0E-10,0.00,null,0E-10,0E-10,0E-10,0E-10,null,0E-10,0E-10,0E-10,null,0E-10,null,null,null,null,null,0E-10,2023-10-25T19:09:20Z,null,null,0E-10,0E-10,0E-10
D-111808,001669,Shakun Goyal,0097165172900 Ext-210,0097126729873,null,null,null,null,null,null,AED,1.0000000,2023-04-25T00:00:00Z,0E-10,0E-10,null,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,null,ADDCORG,ADDC,527768.0000000000,EN,0E-10,11755262673,null,null,null,null,null,1.0000000000,null,null,null,null,2023-04-25T09:22:01Z,null,null,null,null,MAXREG,0E-10,null,null,16.0000000000,REGRETTED,2023-08-03T08:33:20Z,null,0E-10,Small Value Contract,0E-10,0E-10,0E-10,0E-10,001669I3U7,null,null,null,0E-10,null,null,0E-10,null,null,null,null,null,null,null,null,0E-10,0E-10,0.00,null,0E-10,0E-10,0E-10,0E-10,null,0E-10,0E-10,0E-10,null,0E-10,null,null,null,null,null,0E-10,null,null,null,0E-10,0E-10,0E-10
D-111808,001938,Fadi Badran,0097126346900,0097126320478,null,null,null,null,null,null,AED,1.0000000,2023-04-24T00:00:00Z,0E-10,0E-10,2023-08-03T09:55:05Z,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,CDA/PW/0089/Tech./,ADDCORG,ADDC,527723.0000000000

### `quotationline` — priced lines (8 expected, one lump-sum row per vendor)

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline WHERE RFQNUM = 'D-111808' ORDER BY VENDOR

RFQNUM,RFQLINENUM,VENDOR,QUOTATIONLINEID,ITEMNUM,MANUFACTURER,MODELNUM,ORDERQTY,ORDERUNIT,UNITCOST,LINECOST,EOQ,DELIVERYTIME,DELIVERYDATE,ENTERDATE,ENTERBY,ISAWARDED,SELECTEDFORDISPLAY,GLCREDITACCT,TAX1CODE,TAX1,TAX2CODE,TAX2,TAX3CODE,TAX3,TAX4CODE,TAX4,TAX5CODE,TAX5,CATALOGCODE,MEMO,DESCRIPTION,QUOTESTARTDATE,QUOTEENDDATE,LINECOST2,VENDORPACKCODE,VENDORPACKQUANTITY,VENDORWAREHOUSE,SITEID,ORGID,LINETYPE,ITEMSETID,CONDITIONCODE,COMMODITYGROUP,COMMODITY,LANGCODE,HASLD,MKTPLCITEM,ROWSTAMP,AWARDCOST,ISALTAWARDED,ALTAWARDEDLINE,QL1,QL5,QL4,QL3,SUGAWARD,QL2,CONVERSION,LOADEDCOST,ISEBID,ISPIAWARDED,PRICEIMPACT,BOQITEMNUM,APPLYDISCOUNTAFTERBIDS,DISCOUNT_PERCENT,DISCOUNT_UNITCOST,DISCOUNT_APPLIED_AFTERBIDS,LINECOSTWDIS,QL_EXTRA1,WARRANTYDURATION,WARRANTYPERITEM,WARRANTYREQ
D-111808,1.0000000000,001565,2150354795.0000000000,null,null,null,1.00,LS,234651862.0000,234651862.0000,0.00,null,null,2023-05-30T16:50:58Z,001565LGAU,0E-10,1.0000000000,D-0000-00000-251040-00-00000-0-0000,IFTR,11732593.1000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",null,null,null,null,null,null,ADDC,ADDCORG,MATERIAL,null,null,null,null,EN,0E-10,0E-10,12167668254,null,0E-10,null,null,null,null,null,0E-10,QUOTED,null,234651862.0000,0E-10,0E-10,0E-10,null,0E-10,10.07,null,1.0000000000,211014930.0000,null,null,null,null
D-111808,1.0000000000,001938,2150354797.0000000000,null,null,null,1.00,LS,142459514.1000,142459514.1000,0.00,null,null,2023-05-30T17:56:43Z,0019386T4K,1.0000000000,1.0000000000,D-0000-00000-251040-00-00000-0-0000,IFTR,7122975.7050,null,0.00,null,0.00,null,0.00,null,0.00,null,null,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",null,null,null,null,null,null,ADDC,ADDCORG,MATERIAL,null,null,null,null,EN,0E-10,0E-10,12588804553,null,0E-10,null,null,null,null,null,0E-10,QUOTED,null,142459514.1000,0E-10,0E-10,0E-10,null,0E-10,16.94,null,1.0000000000,142459514.1000,null,null,null,null
D-111808,1.0000000000,99102876,2150354799.0000000000,null,null,null,1.00,LS,208545844.6400,208545844.6400,0.00,null,null,2023-05-30T16:35:00Z,99102876Q8UY,0E-10,1.0000000000,D-0000-00000-251040-00-00000-0-0000,IFTR,10427292.2320,null,0.00,null,0.00,null,0.00,null,0.00,null,null,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",null,null,null,null,null,null,ADDC,ADDCORG,MATERIAL,null,null,null,null,EN,0E-10,0E-10,12331963301,null,0E-10,null,null,null,null,null,0E-10,QUOTED,null,208545844.6400,0E-10,0E-10,0E-10,null,0E-10,20.21,null,1.0000000000,166404254.0000,null,null,null,null
D-111808,1.0000000000,99107175,2150354801.0000000000,null,null,null,1.00,LS,195458685.0000,195458685.0000,0.00,null,null,2023-05-31T14:17:41Z,991071756MIW,0E-10,1.0000000000,D-0000-00000-251040-00-00000-0-0000,IFTR,9772934.2500,null,0.00,null,0.00,null,0.00,null,0.00,null,null,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",null,null,null,null,null,null,ADDC,ADDCORG,MATERIAL,null,null,null,null,EN,0E-10,0E-10,12167668303,null,0E-10,null,null,null,null,null,0E-10,QUOTED,null,195458685.0000,0E-10,0E-10,0E-10,null,0E-10,25.59,null,1.0000000000,145438500.0000,null,null,null,null
D-111808,1.0000000000,99443592,2150354803.0000000000,null,null,null,1.00,LS,221525111.0000,221525111.0000,0.00,null,null,2023-05-31T08:58:33Z,99443592NQWF,0E-10,1.0000000000,D-0000-00000-251040-00-00000-0-0000,IFTR,11076255.5500,null,0.00,null,0.00,null,0.00,null,0.00,null,null,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",null,null,null,null,null,null,ADDC,ADDCORG,MATERIAL,null,null,null,null,EN,0E-10,0E-10,12331963298,null,0E-10,null,null,null,null,null,0E-10,QUOTED,null,221525111.0000,0E-10,0E-10,0E-10,null,0E-10,7.31,null,1.0000000000,205329703.0000,null,null,null,null
D-111808,1.000

### `altquotationline` — alternates (0 expected)

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline WHERE RFQNUM = 'D-111808'

TAX1,TAX3,TAX2,ALTQUOTATIONLINEID,QUOTEENDDATE,TAX5,SELECTEDFORDISPLAY,MANUFACTURERNAME,AWARDCOST,ALTQUOTLINEUID,QUOTESTARTDATE,ENTERDATE,UNITCOST,MEMO,ISAWARDED,LINENUM,TAX4,LINECOST2,ENTERBY,DELIVERYDATE,LINECOST,SUGAWARD,SERVICE,GLCREDITACCT,VENDOR,MANUFACTURER,CATALOGCODE,CONVERSION,DELIVERYTIME,MODELNUM,ORDERQTY,EOQ,DESCRIPTION,ITEMNUM,ORDERUNIT,MKTPLCITEM,VENDORPACKCODE,VENDORPACKQUANTITY,VENDORWAREHOUSE,ORGID,QL1,QL2,QL3,QL4,QL5,RFQNUM,RFQLINENUM,SITEID,TAX3CODE,TAX5CODE,TAX4CODE,TAX1CODE,TAX2CODE,ROWSTAMP,HASLD,ITEMSETID,LINETYPE,LOADEDCOST,ISEBID,PRICEIMPACT,BOQITEMNUM,DISCOUNT_PERCENT,APPLYDISCOUNTAFTERBIDS,DISCOUNT_APPLIED_AFTERBIDS,LINECOSTWDIS


### `docinfo` / `doclinks` / `vw_rfqvendor_documents` — attached documents
Never checked for this tender. Schemas unknown — start with `DESCRIBE` (same as `07_trace_bid_with_boq.ipynb` — results should match since it's the same tables).

In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.docinfo

col_name,data_type,comment
DOCUMENT,varchar(50),null
DESCRIPTION,varchar(254),null
APPLICATION,varchar(8),null
STATUS,varchar(8),null
STATUSDATE,timestamp,null
CREATEDATE,timestamp,null
REVISION,"decimal(38,10)",null
CHANGEBY,varchar(31),null
CHANGEDATE,timestamp,null
DOCLOCATION,varchar(10),null


In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.doclinks

col_name,data_type,comment
DOCUMENT,varchar(50),null
OWNERTABLE,varchar(30),null
OWNERID,"decimal(38,10)",null
REFERENCE,varchar(8),null
DOCTYPE,varchar(16),null
DOCVERSION,varchar(20),null
GETLATESTVERSION,"decimal(38,10)",null
CREATEBY,varchar(31),null
CREATEDATE,timestamp,null
CHANGEBY,varchar(31),null


In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents

col_name,data_type,comment
rfqnum,varchar(20),null
vendor,varchar(44),null
bidstatus,varchar(25),null
rfqvendorid,"decimal(38,10)",null
ownertable,varchar(30),null
docinfoid,"decimal(38,10)",null
document,varchar(50),null
urlname,varchar(250),null


In [0]:
%sql
-- SELECT * FROM ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents WHERE RFQNUM = 'D-111808'  -- fix column name once DESCRIBE above shows real columns

## Part 2 — a genuinely `null`-flagged example
Find a `null`-flagged RFQ that actually has substantial `quotationline` data, so the comparison to Part 1 and to `07_trace_bid_with_boq.ipynb`'s `Y` example is fair (not just picking an obscure, genuinely-empty one).

In [0]:
%sql
SELECT r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE,
       COUNT(ql.QUOTATIONLINEID) AS line_count,
       COUNT(DISTINCT ql.VENDOR) AS vendor_count,
       COUNT(DISTINCT ql.BOQITEMNUM) AS distinct_boqitems
FROM ingestion_framework_test.bid_data_exploration.rfq r
LEFT JOIN ingestion_framework_test.bid_data_exploration.quotationline ql ON r.RFQNUM = ql.RFQNUM
WHERE r.DETAILBOQAVAILABLE IS NULL
GROUP BY r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE
ORDER BY line_count DESC
LIMIT 20

RFQNUM,DESCRIPTION,ORGID,ENTERDATE,line_count,vendor_count,distinct_boqitems
N-20585,Blanket Agreement for Valves Repairing and Replacement for all TAQA Transmission Region,TRANSORG,2025-10-31T14:07:38Z,52398,9,0
N-18709,Blanket Agreement for Civil Renovation Work for Power Substation & Water Pumping Stations,TRANSORG,2021-11-17T07:14:05Z,42336,9,0
N-19899,"3 Years Blanket Agreement for Repair of Fire Fighting & Fire Alarm System AD, AA, NE, WR Substations, Pumping Stations and Facilities",TRANSORG,2025-03-03T12:39:07Z,38740,5,7748
N-18530,Blanket Agreement for Valves Repairing and Replacement for all Transco Regions,TRANSORG,2021-07-13T09:30:58Z,33312,6,5550
D-104881,"Execution of Water Connections, Network Relocations & Improvements in Eastern Region",ADDCORG,2015-03-15T14:35:57Z,28544,16,0
D-104888,"Execution of Water Connections, Network Relocations & Improvements in Eastern Region",ADDCORG,2015-03-15T13:33:18Z,28544,16,0
N-18786,Blanket Agreement for Civil Rehabilitation Work for Power Substation & Water Pumping Stations for all regions,TRANSORG,2022-07-28T09:41:41Z,27480,5,0
D-104880,"Execution of Water Connections, Network Relocations & Improvements in Central Region",ADDCORG,2015-02-09T10:14:58Z,19624,11,0
D-104801,Long Term Rate Agreement for Execution of Water Connections Network Relocations and Improvements in Eastern Region- Mussafah Commercial Area Mohamed Bin Zayed City ME(9-12) Al Shabia and Related Island,ADDCORG,2015-01-18T07:25:31Z,17840,10,0
N-18231,"Blanket Agreement for repair of Fire Fighting & Fire Alarm System AD, AA,NE, WR Substations, Pumping Stations and RTU's",TRANSORG,2021-04-07T14:51:47Z,17168,4,4291


Set the RFQNUM to trace below once you've picked one from the candidates above.

In [0]:
dbutils.widgets.text("null_rfqnum", "", "null-flagged RFQNUM to trace")

### `rfq` — header

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq WHERE RFQNUM = :null_rfqnum

RFQNUM,DESCRIPTION,STATUS,STATUSDATE,ENTERDATE,ENTERBY,REPLYDATE,CLOSEONDATE,PURCHASEAGENT,RFQTYPE,REQUIREDDATE,REQUESTEDBY,SHIPTO,SHIPTOATTN,BILLTO,BILLTOATTN,REPLYTO,REPLYTOATTN,FOB,FREIGHTTERMS,SHIPVIA,PAYMENTTERMS,CHANGEBY,CHANGEDATE,PRIORITY,HISTORYFLAG,RFQ1,RFQ2,RFQ3,RFQ4,RFQ5,RFQ6,RFQ7,RFQ8,RFQ9,RFQ10,PRINTDATE,BUYERCOMPANY,ORGID,SITEID,RFQID,LANGCODE,HASLD,TDCOMPLETED,INSURANCEREV,TOTALAWVALUE,PMETHOD,VLSUBMSN,LEGALREV,AWCOMPLETED,TDAGENDA,FINANCEREV,TDMEETING,TDDATE,EXTCLOSEDATE,COMCONT,VLDCSN,WARRANTY,CONTYPE,AWDCSN,CONTPHONE,AWAGENDA,AWDATE,USEREV,FIXEDASSETS,TOTALCOST,TDDECISION,AWDECISION,ESTIMATEDCOST,AWSUBMSN,AWMEETING,DELVDEST,PROJCONS,CURRENCYCODE,INSPCLASS,AFENUM,ESTIMATEVAL,REPNAME,APPTYPE,TYPE,ROWSTAMP,REMARK,WFREPREMARKS,BIDSOPENDATE,KPIREMARKS,MAXINDIVIDUALVOL,TENDERDOCAPPRDATE,TENDERDOCSENDDATE,KPI1DATE,KPI2DATE,KPI3DATE,KPI4DATE,KPI5DATE,KPI6DATE,KPI17ATE,KPI8DATE,KPI9DATE,KPI10DATE,KPI7DATE,KPI11DATE,FIRSTDRAFT,SECONDRAFT,IITPACK,KPI,BIDBOND,PERFBOND,BENEFICIARY,CAPEX_CATEGORY,DURATIONUNIT,FUND_SOURCE,TYPE_BUSINESS,WARRANTYDAY,WARRANTYUNIT,CC,CLARIFICATIONDATE,DEALTBY,FORREVIEW,NOTE,PLEASECOMMENT,PLEASERECYCLE,PLEASEREPLY,PRINTDRAFTCOPY,PRINTESIGNATURE,REFNO,SELECTREPORT,SITEVISIT,TENDERDATE,URGENT,VENDOR,CBIDSOPENDATE,EOICLOSINGDATE,EOIEMAILSUBJECT,EOIEMAILTEXT,EOIREMARKS,EOIUSED,TENDERSTATUS,USEEBIDDING,EOIREFNO,EXTPRICECLOSEDATE,PRICECLOSEDATE,BIDEXTREFNO,INVITEREFNO,PIREFNO,EVALUATIONCUTOFF,TOTALSCORE,TRAINREQ,INSTALLREQ,SAMPLEREQ,CONCREQ,CALCREQ,TESTREPREQ,SPACKINGREQ,TECHDRAWREQ,TRAINING,SPACKDETAILS,TRAININGDAY,TRAININGUNIT,DOWNLOADTYPE,BIDBONDVALUE,EOIPUBLISHDATE,PUBLISHQUOTE,BIDBONDVALIDITY,CEILINGVALUE,RFQAUTHCODE,RFQCOMAUTHCODE,ADVANCEPAYMENT,ADVANCEPAYMENTPERCENT,DETAILBOQAVAILABLE,RETENTIONAPPLIED,RETENTIONPERCENT,COMEVALCUTOFF,COMTOTALSCORE,PARTIALDELIVERY,JUSTIFICATION,CPCSTARTDATE,CPCENDDATE,ECSTARTDATE,ECENDDATE,TECHSTART,TECHEND,COMMSTART,COMMEND,CPCSTART,CPCEND,BLREPDATE,BLSDDATE,AWARDREPDATE,AWARDSDDATE,TOTALAWVALUEWITHTAX,SUBWORKTYPE,CONSREQUIRED,PERFBANK,RETENMONEY,RPAC,RFAC,ADVANCEPAY,INVITEENTERED,INVITETEMP,EOICLOSED,BIDCLOSED,ECSTART,ECEND,COMPDISQUALIFIED,TETC_ENDDATE,AWTC_ENDDATE,TETC_STARTDATE,AWTC_STARTDATE,BUDGETBALANCE,BUDGETAVAILABLE,BUDGETCAT,ESTBUDGET,GUIDNUM,ITRTYPE,PROJECTREV,PROJECTID,ITRNUM,BUDGETCODE,APMASTERREV,APMASTERID,EXT_PROJECTID,EXT_ITRNUM,EXT_BUDGETCODE,EXT_PROJECTREV,POSTBID_DISCOUNT_CLOSEDATE,DISCOUNT_REVISION,EXT_POSTBID_DISCOUNT_CLOSEDATE,RECALLREPUBLISH,ALLOWAMENDMENT,N_PROJECT_NUMBER,N_TASK_NUMBER,UNDERREVIEW,ADPCBOARD,ADPCENDORSE,ADPCEO,ADPCHEC,ADPCREVIEWERS,ADPCTPC,ASTTEAM,ISPROJECT,ESTIMATECOST,ALLOWSINGLEIMPORT,SINGLESOURCE,MEMBERSFORIMP,BP_MASTERMILESTONE,ENDDATE,STARTDATE,PLANNEDSTARTDATE,PROCESSSTARTDATE,AWD_REMARKS,COMEVAL_REMARKS,TECHEVAL_REMARKS,FIRSTCBIDSOPENDATE,SHAREDWITHEWEC,NBFP,NBFPDONE,PROJLIFECYCLEID,EXTAADC1,DURATIONDAYS,CATEGORY,TDCOMMENTS,TDENDORSEDATE,TDENTERBY,TDRESULT,COMOPENDATE,COMOPENUSER,STAGE,TECHOPENDATE,TECHOPENUSER,AUTOROUTE,BUDGENHCOMMENTS,BUDGENHENDORSEDATE,BUDGENHENTERBY,BUDGENHRESULT,BLACOMMENTS,BLAENDORSEDATE,BLATDENTERBY,BLARESULT,BUDGENHFMDATE,BUDGENHFMENTERBY,BUDGENHMDDATE,BUDGENHMDENTERBY,BUDGENHSMDATE,BUDGENHSMENTERBY,BLASMDATE,BLASMENTERBY,SAMPLETESTREQ,BUDGENHCOMMENTSFIN,BUDGENHFINOFF,BUDGENHRESULTFIN,BUDGENHDEPTDEC,BUDGENHFMDEC,BUDGENHMDDEC,BUDGENHREJBY,BUDGENHREJDATE,BUDGENHREJREAASON,BUDGENHSMDEC,PROJECTDUR,PROJECTDURUNIT,TD,FU_STATUS,FU_SOURCE,FU_STATUSBY,FU_STATUSDATE,TECHEVALREQ,N_ISTTH


### `rfqvendor`

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor WHERE RFQNUM = :null_rfqnum ORDER BY VENDOR

RFQNUM,VENDOR,CONTACT,PHONE,FAXPHONE,EMAIL,CUSTOMERNUM,FOB,FREIGHTTERMS,SHIPVIA,PAYMENTTERMS,CURRENCYCODE,EXCHANGERATE,EXCHANGEDATE,BUYAHEAD,INTERNAL,REPLIEDDATE,GLCREDITACCT,INCLUSIVE1,INCLUSIVE2,INCLUSIVE3,INCLUSIVE4,INCLUSIVE5,VENDORQUOTENUM,ORGID,SITEID,RFQVENDORID,LANGCODE,HASLD,ROWSTAMP,BUYERREMARK,LINENUM,RFQV1,PRODUCTGROUPID,CURRENCYCODE2,RFQV3,RFQV4,DISCOUNT,SQR,MNGREMARK,ENTERDATE,RFQV2,OTHERCHG,RFQV5,SSUPREMARK,ENTEREDBY,ISVENINC,BYRRMK,SNSRMK,MGTRMK,BIDSTATUS,BIDSTATUSDATE,PASSWORD,PRICEIMPACT,REGRETREASON,RESETPASSWORD,CI,TCI,TI,USERID,VALIDTO,SUPPORTREMARKS,SUPPOERTCONTACTED,CIINITIATED,VALIDITYPERIOD,SCORE,SURVEYCREATED,BASETOTALAWARDCOST,TOTALAWARDCOST,TOTALDISCOUNT,TOTAWDWODISCOUNT,TOTBIDCOSTWDIS,TOTBIDCOSTWODIS,TOTALAWARDCOSTWITHTAX,COMPEVALTYPE,APPLYDISCOUNTBEFOREBIDS,APPLYDISCOUNTAFTERBIDS,DISCOUNT_PERCENT,APPLYDISCOUNT,DISCOUNTAFTERBIDS,APPLYDISCOUNTAFTERPI,DCI,DCIREQUIRED,DISCOUNTSTATUS,DISCOUNT_APPLIED_AFTERBIDS,DISCOUNT_APPLIED_AFTERPI,DISCOUNT_APPLIED_BEFOREBIDS,POSTBID_DISCOUNT_COUNTER,POSTBID_DISCOUNT_SENT,DISCOUNT_REVISION,DISCOUNT_SUBMISSION_DATE,TOTALAWARDCOSTWDIS,TOTALAWARDCOSTWITHTAXWDIS,DISCOUNT_APPLY_DATE,QUOTERENEWED,QUOTEVALIDITY,RFQV_EXTRA1,RFQV_EXTRA2,ISAWARDED,AGREECOC,WAIVEOWNERVAL


### `quotationline`

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline WHERE RFQNUM = :null_rfqnum ORDER BY VENDOR, BOQITEMNUM, RFQLINENUM

RFQNUM,RFQLINENUM,VENDOR,QUOTATIONLINEID,ITEMNUM,MANUFACTURER,MODELNUM,ORDERQTY,ORDERUNIT,UNITCOST,LINECOST,EOQ,DELIVERYTIME,DELIVERYDATE,ENTERDATE,ENTERBY,ISAWARDED,SELECTEDFORDISPLAY,GLCREDITACCT,TAX1CODE,TAX1,TAX2CODE,TAX2,TAX3CODE,TAX3,TAX4CODE,TAX4,TAX5CODE,TAX5,CATALOGCODE,MEMO,DESCRIPTION,QUOTESTARTDATE,QUOTEENDDATE,LINECOST2,VENDORPACKCODE,VENDORPACKQUANTITY,VENDORWAREHOUSE,SITEID,ORGID,LINETYPE,ITEMSETID,CONDITIONCODE,COMMODITYGROUP,COMMODITY,LANGCODE,HASLD,MKTPLCITEM,ROWSTAMP,AWARDCOST,ISALTAWARDED,ALTAWARDEDLINE,QL1,QL5,QL4,QL3,SUGAWARD,QL2,CONVERSION,LOADEDCOST,ISEBID,ISPIAWARDED,PRICEIMPACT,BOQITEMNUM,APPLYDISCOUNTAFTERBIDS,DISCOUNT_PERCENT,DISCOUNT_UNITCOST,DISCOUNT_APPLIED_AFTERBIDS,LINECOSTWDIS,QL_EXTRA1,WARRANTYDURATION,WARRANTYPERITEM,WARRANTYREQ


### `altquotationline`

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline WHERE RFQNUM = :null_rfqnum ORDER BY VENDOR, RFQLINENUM

TAX1,TAX3,TAX2,ALTQUOTATIONLINEID,QUOTEENDDATE,TAX5,SELECTEDFORDISPLAY,MANUFACTURERNAME,AWARDCOST,ALTQUOTLINEUID,QUOTESTARTDATE,ENTERDATE,UNITCOST,MEMO,ISAWARDED,LINENUM,TAX4,LINECOST2,ENTERBY,DELIVERYDATE,LINECOST,SUGAWARD,SERVICE,GLCREDITACCT,VENDOR,MANUFACTURER,CATALOGCODE,CONVERSION,DELIVERYTIME,MODELNUM,ORDERQTY,EOQ,DESCRIPTION,ITEMNUM,ORDERUNIT,MKTPLCITEM,VENDORPACKCODE,VENDORPACKQUANTITY,VENDORWAREHOUSE,ORGID,QL1,QL2,QL3,QL4,QL5,RFQNUM,RFQLINENUM,SITEID,TAX3CODE,TAX5CODE,TAX4CODE,TAX1CODE,TAX2CODE,ROWSTAMP,HASLD,ITEMSETID,LINETYPE,LOADEDCOST,ISEBID,PRICEIMPACT,BOQITEMNUM,DISCOUNT_PERCENT,APPLYDISCOUNTAFTERBIDS,DISCOUNT_APPLIED_AFTERBIDS,LINECOSTWDIS


### documents
Reuse the schemas discovered in Part 1 (same tables) — just filter:

In [0]:
%sql
-- SELECT * FROM ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents WHERE RFQNUM = :null_rfqnum  -- fix column name once Part 1's DESCRIBE shows real columns

## Observations — fill in once both parts have real results

| | D-111808 (`N`/unconfirmed) | `null` example | `Y` example (`07`) |
|---|---|---|---|
| `rfq.DETAILBOQAVAILABLE` actual value | | | Y |
| `quotationline` row count | 8 (known) | | |
| Itemized (`BOQITEMNUM` populated, multiple per vendor)? | No (known) | | |
| Documents actually attached? | | | |

_(fill in blanks once run — this table is the real answer to "does `DETAILBOQAVAILABLE` mean what we think")_